In [1]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

import pandas as pd
import psycopg2

from typing import Tuple

In [ ]:
def import_data() -> pd.DataFrame:
    """
    Import data from Housing_Price database into a pandas DataFrame.

    Returns:
        df (pd.DataFrame): The imported table from the database.
    """
    conn = psycopg2.connect(dsn = "host='host name' port='port' " \
    "dbname=Housing_Price " \
    "user=postgres " \
    "password='password'")
    df = pd.read_sql(sql = "SELECT * FROM clean_data_housing;", con = conn)
    conn.close()

    return df

Now we compute the R2-score, MSE, MAE, RMSE for both training and test data.

In [3]:
def model_evaluation(model, X: pd.DataFrame, y: pd.DataFrame) -> tuple:
    """
    Calculate the r2-score, Mean Squared Error, Mean Absolute Error, Root Mean Squared Error for
    the given model.

    Args:
        model:
            A trained model.
        X (pd.DataFrame):
            Features.
        y (pd.DataFrame):
            True labels.
        
    Returns:
        Tuple:
            - R2-score
            - Mean Squared Error
            - Mean Absolute Error
            - Room Mean Squared Error
    """
    predictions = model.predict(X)
    r2_score = model.score(X, y)
    mse = mean_squared_error(y_pred = predictions, y_true = y)
    mae = mean_absolute_error(y_pred = predictions, y_true = y)
    rmse = root_mean_squared_error(y_pred = predictions, y_true = y)

    return r2_score, mse, mae, rmse

`LinearRegression` minimizes the Mean Squared Error by solving the closed-form normal equation.

#### Objective(minimize MSE):
$$
\min_{w}||y - Xw||^{2}
$$

#### Closed-from solution:
$$
w = (X^{T}X)^{-1}X^{T}y
$$

#### bias:
$$
b = \bar y - \bar X.w
$$

In [4]:
df = import_data()

X = df.iloc[:, 0:-1]
y = df.iloc[:, -1]
# Split the data into training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 50)

# Train the LinearRegression model
linear_reg = LinearRegression().fit(X_train, y_train)

/tmp/ipykernel_1056/302948749.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql = "SELECT * FROM clean_data_housing;", con = conn)


In [5]:
train_r2_score, train_mse, train_mae, train_rmse = model_evaluation(model = linear_reg, X = X_train, y = y_train)
test_r2_score, test_mse, test_mae, test_rmse = model_evaluation(model = linear_reg, X = X_test, y = y_test)

print(f"Training scores:\n\
        R2-score = {train_r2_score}     MSE = {train_mse}     MAE = {train_mae}     RMSE = {train_mae}\n\n\
Test Scores:\n\
        R2-score = {test_r2_score}     MSE = {test_mse}     MAE = {test_mae}     RMSE = {test_rmse}\n")


Training scores:
        R2-score = 0.9348197061052161     MSE = 623157782.3354175     MAE = 19939.043375430876     RMSE = 19939.043375430876

Test Scores:
        R2-score = 0.9341827311750155     MSE = 624571835.5170696     MAE = 19924.234818437013     RMSE = 24991.435243240223



`SGDRegressor` uses Stochastic Gradient Descent instead of the closed-form solution.
It minimizes MSE:
$$
L = \frac{1}{2}(y - \hat y)^{2} + \Omega
$$

Weights and Bias update:
$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}
$$
$$
b \leftarrow b - \eta \frac{\partial L}{\partial b}
$$

In [6]:
from sklearn.preprocessing import StandardScaler


models = {
    # SGDRegressor with L1 regularization
    "L1": SGDRegressor(loss = "squared_error", max_iter = 1000,
                        penalty = "l1", random_state = 50, learning_rate = "constant", eta0 = 0.001),
    # SGDregressor with L2 regularization
    "L2": SGDRegressor(loss = "squared_error", max_iter = 1000,
                        penalty = "l2", random_state = 50, learning_rate = "constant", eta0 = 0.001),
    # SGDregressor with Elastic net regularization
    "elasticnet": SGDRegressor(loss = "squared_error", max_iter = 1000,
                        penalty = "elasticnet", random_state = 50, learning_rate = "constant", eta0 = 0.001),
    # SGDregressor with no regularization
    "no_reg": SGDRegressor(loss = "squared_error", max_iter = 1000,
                        penalty = None, alpha = 0, random_state = 50, learning_rate = "constant", eta0 = 0.001)
}

# Transform the Features otherwise the models will diverge.
scaler = StandardScaler()
X_train_transformed = scaler.fit_transform(X_train)
X_test_transformed = scaler.transform(X_test)

for name, model in models.items():
    model.fit(X_train_transformed, y_train)
    scores = model_evaluation(model, X_test_transformed, y_test)
    print(f"model: {name}\n\
                R2-score = {scores[0]}     MSE = {scores[1]}     MAE = {scores[2]}     RMSE = {scores[3]}")






model: L1
                R2-score = 0.9339234016505766     MSE = 627032738.5592972     MAE = 19963.318112490753     RMSE = 25040.62176862422
model: L2
                R2-score = 0.9339225392955879     MSE = 627040921.8620663     MAE = 19963.440137190584     RMSE = 25040.7851686417
model: elasticnet
                R2-score = 0.9339226698111166     MSE = 627039683.3372108     MAE = 19963.421835044093     RMSE = 25040.76043847732
model: no_reg
                R2-score = 0.9339234016914116     MSE = 627032738.1717944     MAE = 19963.31810730443     RMSE = 25040.621760886737
